# Japan-Paw: indexación por tandas con estado en Drive

Usa este cuaderno con la versión corregida del indexador. Primero procesa hasta **100 archivos o 15 minutos**, con dos trabajadores. No lanza el catálogo entero ni publica resultados. Ejecuta una sola sesión por directorio de estado.

Los fallos de red quedan separados de los archivos incompatibles. La comprobación de una muestra de piezas no equivale a verificar todos los bytes del video. Consulta `docs/COLAB.md` para interpretar estados y reanudar.

Si no se prepara ningún archivo, revisa la **etapa, el host y el código HTTP** del diagnóstico. Un total de diferidos no demuestra un bloqueo general de Colab. Los fallos al buscar, descargar un torrent y leer piezas del video tienen causas distintas. No actives reintentos masivos ni elimines pausas sin revisar esa evidencia.


In [ ]:
# 1. Obtener el código corregido. Sube el ZIP entregado junto a este cuaderno.
# Si ya publicaste las correcciones en GitHub, puedes cambiar SOURCE_MODE a "git".
from pathlib import Path
import json, os, shutil, subprocess, zipfile, hashlib, signal, datetime

SOURCE_MODE = "upload"  # "upload" o "git"
UPDATE_FROM_GIT = False  # True solo cuando quieras actualizar explícitamente una copia Git existente.
PROJECT = Path("/content/ExtenJap")
REPOSITORY = "https://github.com/JuanPerezC893/ExtenJap.git"

if SOURCE_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise RuntimeError("Sube exactamente un ZIP del proyecto corregido.")
    if PROJECT.exists():
        raise RuntimeError("La carpeta /content/ExtenJap ya existe. Reutiliza las siguientes celdas o cambia PROJECT a una carpeta nueva; no se sobrescribe automáticamente.")
    with zipfile.ZipFile(archives[0]) as archive:
        # El bundle debe contener package.json y raw-catalog.json en su raíz.
        names = archive.namelist()
        if "package.json" not in names or "raw-catalog.json" not in names:
            raise RuntimeError("El ZIP debe contener package.json y raw-catalog.json en la raíz.")
        root = PROJECT.resolve()
        for member in archive.infolist():
            target = (root / member.filename).resolve()
            if not target.is_relative_to(root):
                raise RuntimeError("El ZIP contiene una ruta fuera de la carpeta destino.")
        PROJECT.mkdir(parents=True)
        archive.extractall(PROJECT)
elif SOURCE_MODE == "git":
    if not PROJECT.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(PROJECT)], check=True)
    elif UPDATE_FROM_GIT:
        local_changes = subprocess.check_output(["git", "status", "--porcelain"], cwd=PROJECT, text=True)
        if local_changes.strip():
            raise RuntimeError("Hay cambios locales en el proyecto. Se conservan: revisa o respalda esos cambios antes de actualizar; no se borra nada automáticamente.")
        subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=PROJECT, check=True)
    else:
        print("Se conserva la versión Git existente. Para actualizarla, activa UPDATE_FROM_GIT en esta celda.")
else:
    raise ValueError("SOURCE_MODE debe ser upload o git.")

source = (PROJECT / "indexer.mjs").read_text()
if "state-dir" not in source or "max-minutes" not in source:
    raise RuntimeError("Esta copia del indexador no contiene las opciones de reanudación. Carga la versión corregida antes de continuar.")
node_version = subprocess.check_output(["node", "--version"], text=True).strip()
print("Node:", node_version)
if int(node_version.lstrip("v").split(".")[0]) < 20:
    raise RuntimeError("Este flujo necesita Node.js 20 o posterior. Actualiza Node antes de instalar dependencias.")
subprocess.run(["npm", "ci"], cwd=PROJECT, check=True)
print("Código preparado en", PROJECT)


In [ ]:
# 2. Montar Drive y conservar las asociaciones existentes, solo al iniciar un estado nuevo.
from google.colab import drive

drive.mount("/content/drive")
STATE = Path("/content/drive/MyDrive/JapanPaw-index")
CATALOG = PROJECT / ("raw-catalog.json" if (PROJECT / "raw-catalog.json").exists() else "dist/indexed-catalog.json")
STATE.mkdir(parents=True, exist_ok=True)

registry = STATE / "verified-matches.json"
job_state = STATE / "indexer-state.json"
if registry.exists() or job_state.exists():
    print("Estado existente: se conserva sin sustituir por el del repositorio.")
else:
    # Primero se copian los torrents. Instalar el registro es el último paso.
    source_torrents = PROJECT / "dist/torrents"
    target_torrents = STATE / "dist/torrents"
    target_torrents.mkdir(parents=True, exist_ok=True)
    for path in source_torrents.glob("*.torrent"):
        destination = target_torrents / path.name
        if not destination.exists():
            shutil.copy2(path, destination)
    source_registry = PROJECT / "verified-matches.json"
    if source_registry.exists():
        json.loads(source_registry.read_text())  # No instalar JSON incompleto.
        temporary = STATE / "verified-matches.bootstrap.tmp"
        shutil.copy2(source_registry, temporary)
        os.replace(temporary, registry)
    print("Directorio de estado inicializado.")

# Guardar la entrada por su huella permite saber qué catálogo usó cada tanda.
catalog_hash = hashlib.sha256(CATALOG.read_bytes()).hexdigest()
input_copy = STATE / "inputs" / ("catalog-" + catalog_hash + ".json")
input_copy.parent.mkdir(parents=True, exist_ok=True)
if not input_copy.exists():
    shutil.copy2(CATALOG, input_copy)
try:
    revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT, text=True, stderr=subprocess.DEVNULL).strip()
except subprocess.CalledProcessError:
    revision = "bundle local; conservar el ZIP del código"
metadata = {"recordedAt": datetime.datetime.now(datetime.timezone.utc).isoformat(), "catalogSha256": catalog_hash, "codeRevision": revision, "node": node_version}
(STATE / "inputs" / ("run-input-" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S") + ".json")).write_text(json.dumps(metadata, indent=2))
print("Estado persistente:", STATE)
print("No ejecutes otro indexador simultáneo sobre esta misma carpeta.")


In [ ]:
# 2b. (Recomendado para Colab) Activar Cloudflare WARP y puente HTTP local (Privoxy).
# Evita el bloqueo HTTP 403 de datacenter en Craftervault sin desconectar la sesion de Colab.
# Configura el tunel en modo proxy SOCKS5 y crea el puente en http://127.0.0.1:8118.
import subprocess, time
from pathlib import Path

print("1. Instalando Cloudflare WARP y Privoxy...")
commands = """
curl -fsSL https://pkg.cloudflareclient.com/pubkey.gpg | sudo gpg --yes --dearmor --output /usr/share/keyrings/cloudflare-warp-archive-keyring.gpg
echo "deb [signed-by=/usr/share/keyrings/cloudflare-warp-archive-keyring.gpg] https://pkg.cloudflareclient.com/ $(lsb_release -cs) main" | sudo tee /etc/apt/sources.list.d/cloudflare-client.list
sudo apt-get update -qq && sudo apt-get install -y -qq cloudflare-warp privoxy
"""
subprocess.run(commands, shell=True, check=True)

print("2. Iniciando servicio WARP en segundo plano...")
subprocess.Popen(["warp-svc"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

print("3. Registrando y conectando WARP en modo Proxy...")
subprocess.run("warp-cli --accept-tos registration new 2>/dev/null || warp-cli --accept-tos register", shell=True)
subprocess.run("warp-cli --accept-tos mode proxy 2>/dev/null || warp-cli --accept-tos set-mode proxy", shell=True)
subprocess.run("warp-cli --accept-tos connect", shell=True, check=True)
time.sleep(3)

print("4. Configurando puente HTTP a SOCKS5 con Privoxy...")
with open("/etc/privoxy/config", "a") as f:
    f.write("\nforward-socks5 / 127.0.0.1:40000 .\n")
subprocess.run(["service", "privoxy", "restart"], check=True)

PROXY_FILE = Path("/content/proxies.txt")
PROXY_FILE.write_text("http://127.0.0.1:8118\n")
print(f"Proxy local listo y guardado en {PROXY_FILE}")

print("\n--- Comprobando tunel WARP ---")
subprocess.run("curl -x http://127.0.0.1:8118 -s https://cloudflare.com/cdn-cgi/trace | grep -E 'warp|ip'", shell=True)


## 2c. (Opcional) Test de límites y estrés (Benchmark)

Ejecuta esta celda si deseas encontrar la **máxima velocidad segura** antes de lanzar tandas largas.
Prueba automáticamente perfiles desde 2 trabajadores / 400 ms hasta 16 trabajadores / 0 ms en tandas de 25 archivos.
Si los buscadores responden con **HTTP 429 (límite de tasa)** o pausas, el test **se detiene al instante** para proteger tus cuotas.
**Todo el trabajo realizado durante la prueba se guarda en Drive** (no se pierde nada).
Al finalizar, te mostrará una tabla con el rendimiento de cada nivel y te indicará el **Punto Óptimo Recomendado** para configurar en la celda 3.


In [ ]:
# 2c. Ejecutar Benchmark de límites y estrés
from pathlib import Path
import os, subprocess, signal, json

# Asegurar variables base si el kernel fue reiniciado
if "PROJECT" not in globals():
    PROJECT = Path("/content/ExtenJap")
if "STATE" not in globals():
    STATE = Path("/content/drive/MyDrive/JapanPaw-index")
if "CATALOG" not in globals():
    CATALOG = PROJECT / ("raw-catalog.json" if (PROJECT / "raw-catalog.json").exists() else "dist/indexed-catalog.json")

BATCH_SIZE = 25  # Archivos a procesar por cada nivel de prueba
SERIES = []  # Opcional: filtra una o más series (ej: ["185874"] o ["Mahou Shoujo"])
USE_PROXIES = True  # Recomendado con WARP (/content/proxies.txt)
PROXY_FILE = Path("/content/proxies.txt")

try:
    current_rev = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=PROJECT, text=True).strip()
    print(f"Versión de código en {PROJECT}: {current_rev}")
except Exception:
    pass

if not (PROJECT / "benchmark-limits.mjs").is_file():
    raise RuntimeError("No se encontró benchmark-limits.mjs en " + str(PROJECT) + ". Activa UPDATE_FROM_GIT = True en la celda 1 para actualizar.")

cmd = ["node", "benchmark-limits.mjs", "--catalog", str(CATALOG), "--state-dir", str(STATE), "--batch-size", str(BATCH_SIZE)]
if SERIES:
    cmd.extend(["--series", *map(str, SERIES)])
if USE_PROXIES and PROXY_FILE.is_file():
    cmd.extend(["--proxy", "--proxy-file", str(PROXY_FILE)])

print("Comando:", " ".join(cmd))
process = subprocess.Popen(cmd, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1, start_new_session=True)
try:
    for line in process.stdout:
        print(line, end="")
    BENCH_EXIT = process.wait()
except KeyboardInterrupt:
    print("\nInterrupción solicitada...")
    if process.poll() is None:
        os.killpg(process.pid, signal.SIGINT)
    try:
        remaining, _ = process.communicate(timeout=30)
        if remaining:
            print(remaining, end="")
        BENCH_EXIT = process.returncode
    except subprocess.TimeoutExpired:
        pass


In [ ]:
# 3. Ejecutar una tanda. Puedes repetir esta celda o activar AUTO_LOOP para continuar.
from pathlib import Path
import os, subprocess, signal, json, time

# Asegurar variables base y montaje por si el kernel de Python fue reiniciado
if "PROJECT" not in globals():
    PROJECT = Path("/content/ExtenJap")
if "STATE" not in globals():
    STATE = Path("/content/drive/MyDrive/JapanPaw-index")
if "CATALOG" not in globals():
    CATALOG = PROJECT / ("raw-catalog.json" if (PROJECT / "raw-catalog.json").exists() else "dist/indexed-catalog.json")
if not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass

# --- PARÁMETROS DE EJECUCIÓN (Aumentados para mayor rendimiento) ---
CONCURRENCY = 12       # Nivel 5: 12 trabajadores concurrentes con WARP
LIMIT = 2000           # Tamaño de tanda ampliado (ej: 2000 archivos)
MAX_MINUTES = 60       # Tiempo máximo por tanda en minutos (ej: 60, 90 o 120)
INTERVAL_MS = 30       # Espaciado rápido de 30 ms
AUTO_LOOP = False      # Si es True, encadena tandas automáticamente sin detenerse
MAX_LOOPS = 10         # Cantidad máxima de tandas continuas si AUTO_LOOP está activo
SERIES = []            # Opcional: filtra una o más series (ej: ["185874"])
RETRY_PENDING = False  # Reintento explícito cuando corresponda; conserva las pausas guardadas.
FORCE_LOCK = False     # Activar sólo si necesitas forzar la remoción de indexer.lock
USE_PROXIES = True     # Activado con el tunel local WARP (/content/proxies.txt)
PROXY_FILE = Path("/content/proxies.txt")

# Limpieza automática preventiva de indexer.lock si pertenece a una sesión interrumpida anterior
lock_file = STATE / "indexer.lock"
if lock_file.is_file():
    try:
        lock_data = json.loads(lock_file.read_text())
        lock_pid = lock_data.get("pid")
        is_running = False
        if lock_pid:
            try:
                os.kill(lock_pid, 0)
                is_running = True
            except OSError:
                is_running = False
        if not is_running:
            print(f"Aviso: Limpiando indexer.lock residual de sesión anterior (PID {lock_pid} no activo)...")
            lock_file.unlink(missing_ok=True)
        elif FORCE_LOCK:
            print("Aviso: FORCE_LOCK activo. Forzando eliminación de indexer.lock...")
            lock_file.unlink(missing_ok=True)
    except Exception:
        if FORCE_LOCK:
            lock_file.unlink(missing_ok=True)

# Esta celda usa la copia preparada; no actualiza ni restablece Git.
try:
    current_rev = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=PROJECT, text=True).strip()
    print(f"Versión de código en {PROJECT}: {current_rev}")
except Exception:
    pass

loop_iteration = 0
while True:
    loop_iteration += 1
    if AUTO_LOOP:
        print(f"\n{'='*60}\n   INICIANDO TANDA {loop_iteration} / {MAX_LOOPS} (Modo Continuo)\n{'='*60}\n")

    command = ["node", "indexer.mjs", "--catalog", str(CATALOG), "--state-dir", str(STATE),
               "--concurrency", str(CONCURRENCY), "--limit", str(LIMIT), "--max-minutes", str(MAX_MINUTES)]
    if INTERVAL_MS is not None:
        command.extend(["--interval-ms", str(INTERVAL_MS)])
    if SERIES:
        command.extend(["--series", *map(str, SERIES)])
    if RETRY_PENDING:
        command.append("--retry-pending")
    if FORCE_LOCK:
        command.append("--force-lock")
    if USE_PROXIES:
        if not PROXY_FILE.is_file():
            raise RuntimeError("USE_PROXIES está activo pero no existe el archivo de proxies. Proporciona el archivo o desactiva la opción explícitamente.")
        command.extend(["--proxy", "--proxy-file", str(PROXY_FILE)])

    print("Comando:", " ".join(command))
    process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end="")
        INDEXER_EXIT = process.wait()
    except KeyboardInterrupt:
        print("\nInterrupción solicitada. Esperando el guardado del indexador...")
        if process.poll() is None:
            os.killpg(process.pid, signal.SIGINT)
        try:
            remaining, _ = process.communicate(timeout=45)
            if remaining:
                print(remaining, end="")
            INDEXER_EXIT = process.returncode
        except subprocess.TimeoutExpired:
            raise RuntimeError("El proceso sigue cerrando. No inicies otra tanda ni cierres el entorno; revisa el proceso antes de continuar.")
        break

    # Node puede informar 130 o ser terminado directamente por SIGINT (-2).
    if INDEXER_EXIT == -signal.SIGINT:
        INDEXER_EXIT = 130
    messages = {0: "Tanda terminada; revisar cuántos archivos se prepararon.",
                2: "Trabajo diferido por proveedores; revisar pausas antes de reintentar.",
                130: "Interrumpido; conservar y revisar el último punto de control."}
    print("\nSalida:", INDEXER_EXIT, messages.get(INDEXER_EXIT, "Error fatal: revisar la salida antes de continuar."))
    if INDEXER_EXIT not in (0, 2, 130):
        raise RuntimeError("El indexador terminó con un error fatal. No compilar automáticamente.")
    if not AUTO_LOOP or INDEXER_EXIT != 0 or loop_iteration >= MAX_LOOPS:
        break
    print(f"\nTanda completada con éxito. Esperando 5 segundos antes de la siguiente tanda...\n")
    time.sleep(5)


In [ ]:
# 4. Compilar los avances guardados. No declara exitosos los archivos pendientes.
from pathlib import Path
import subprocess
if "PROJECT" not in globals():
    PROJECT = Path("/content/ExtenJap")
if "STATE" not in globals():
    STATE = Path("/content/drive/MyDrive/JapanPaw-index")
if "INDEXER_EXIT" not in globals():
    INDEXER_EXIT = 0
if INDEXER_EXIT not in (0, 2, 130):
    raise RuntimeError("Revisa el fallo del indexador antes de compilar.")
subprocess.run(["node", "build.mjs", "--state-dir", str(STATE)], cwd=PROJECT, check=True)
print("Publicación preparada en", STATE / "dist")


## Exportación opcional

Drive contiene el estado de trabajo. Esta última celda crea una copia adicional descargable de **todo el directorio de estado**, incluidos las pausas de proveedores, los registros y los torrents. No sube nada a GitHub. Conserva también el ZIP del código; al restaurar, usa una carpeta nueva y selecciona esa carpeta como `STATE`.

El montaje de Drive puede tardar en sincronizar. Una sesión eliminada de forma abrupta puede perder el trabajo posterior al último guardado disponible. Detén la tanda antes de exportar y no ejecutes otro escritor al mismo tiempo.


In [ ]:
# 5. Descargar un respaldo después de cerrar el indexador.
if "process" in globals() and process.poll() is None:
    raise RuntimeError("El indexador todavía está activo; termina la tanda antes de exportar.")
from google.colab import files
stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
archive_base = Path("/content") / ("JapanPaw-estado-" + stamp)
archive_file = shutil.make_archive(str(archive_base), "zip", root_dir=STATE)
print("Respaldo:", archive_file)
files.download(archive_file)
